In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from robot_wm.inference.actor.base import RobotActionHistory, RobotObsHistory
from robot_wm.inference.actor.cost.visual_based_latents_cost import \
    L2VisualLatentsCost
from robot_wm.inference.robot.base import RobotObs
from robot_wm.inference.task.reference_episode import ImageProprioGoal
from robot_wm.utils.config import from_config

## Dataset

We start by loading the droid dataset

In [ ]:
dataset = from_config(
    "datasets/configs/datasets/droid.yaml",
    # can also use RAD_paths.csv for the heldout set.
    overrides=[
        "manifest=/fsx-cortex-datacache/shared/datasets/droid/011825/droid_h5/train_paths.csv"
    ],
)

#### Converting Droing to RobotObs and Actions

For inference we use `RobotObsHistory`, `RobotActionHistory` as abstractions to handle sensor streams, the following is a minimal code to convert the droid dataset used for training to this format. 

> We should consider also making this an util or a default on the inference code

In [ ]:
sample = dataset[1000]
obs = sample["episode_data"]["observation"]

observation_history = RobotObsHistory(max_context=1000, freq=30)
for i in range(len(obs["joint_position"])):
    joints = obs["joint_position"][i]

    ee_pose = obs["cartesian_position"][i]
    gripper = obs["gripper_position"][i]
    ee_pose_with_gripper = np.concatenate((ee_pose, gripper))

    cam = obs["exterior_image_1_left"][i]
    cam = np.transpose(cam, (2, 0, 1)) / 255.0

    robot_obs = RobotObs(joints, ee_pose_with_gripper, cam)
    observation_history.push(robot_obs)

action_history = observation_history.get_action_history_from_ee_deltas()
observation_history.show()

## World Model
Instantiate a world model from our model zoo or _menagerie_. 

> Each world model requires you to have setup their corresponding project, see corresponding README's or Cortex Wiki for more details.

In [ ]:
# can add overrides to load different weights or change params

# wm = from_config('menagerie/config/st_wm.yaml')
# wm = from_config('menagerie/config/dino_wm.yaml')
wm = from_config("menagerie/config/jepa_wm.yaml")

#### Does the world model encode correctly an observation stream?

We encode observations from droid, no rollouts. This should look very similar to the dataset video.

In [ ]:
# Some models don't allow a very large context for rollouts, you could see and change that parameter for this visualization at wm.max_context
wm.max_context = 1000
ans = wm.encode_history(observation_history, action_history)
imgs = wm.decode_latents(ans)
imgs.show()

#### Does the WM does sensible future predictions?
Now encode a small history and make predictions predictions using GT actions.

In [ ]:
init, context, end = 0, 20, -1

# Take 40 frames of context
context_obs = observation_history[init : init + context]
context_actions = context_obs.get_action_history_from_ee_deltas()

# Encode them
history = wm.encode_history(context_obs, context_actions)

# Take all the future GT actions
future_obs = observation_history[init + context :]  # can be used for comparison
future_actions = future_obs.get_action_history_from_ee_deltas()

# Rollout and show them
rollout = wm.rollout(history, future_actions)
pred_imgs = wm.decode_latents(rollout)
pred_imgs.show(), future_obs.show()

## Planning with a World Model

Cool, we have a WM that can predict the future. Now let's try to plan with it.

#### Goal and Cost

In [ ]:
## Let's create our goal as the last image in the droid episode we got above
last_img = obs["exterior_image_1_left"][-1]
last_img = np.transpose(last_img, (2, 0, 1)) / 255.0
goal = ImageProprioGoal(
    last_img,
    np.concatenate((obs["cartesian_position"][-1], obs["gripper_position"][-1])),
)

## And encode it
encoded_goal = wm.encode_goal(goal)

## Lets also create a cost function
cost = L2VisualLatentsCost()

#### Does the cost function work at all?
We start by computing the cost between the goal (last frame) and the full history of latents. Ideally this should decrease and be minimal at the last frame. It will not always monotonically decrease, and that's expected

> Rarely goes to zero, minimal imperceptible differences between frames imply a large cost.

In [ ]:
gt_encoded_latents = wm.encode_history(observation_history, action_history)
gt_latents_cost = cost(encoded_goal, gt_encoded_latents)
plt.plot(gt_latents_cost[0].float().cpu().numpy())

#### Can we use this cost function with predictions?
Instead of using the encoded history, we can use the predictions from the world model. Should look similar as above but is a more challenging setting because our predictions might deviate from the GT.

> Depending on the encode this might be good/bad. So far it seems that Dino is the best encoder for this task and Cosmos the worst. Jepa in the middle

In [ ]:
history = wm.encode_history(context_obs, context_actions)
rollout_latents = wm.rollout(history, future_actions)
rollout_latents_cost = cost(encoded_goal, rollout_latents)
plt.plot(rollout_latents_cost[0].float().cpu().numpy())

In [ ]:
decoded_latents = wm.decode_latents(rollout_latents)
best_frame = np.argmin(rollout_latents_cost[0].float().cpu().numpy())
best_frame = decoded_latents[int(best_frame)]
best_frame.show(), goal.show()

#### Planning
We now have all the ingredients to plan. For the planner we'll make an easier task, try to plan 1 second in the future

In [ ]:
init, context, end = 0, 30, 60

# Take a few frames of context
context_obs = observation_history[init : init + context]
context_actions = context_obs.get_action_history_from_ee_deltas()

# Use the "end" as goal
goal = ImageProprioGoal(
    np.transpose(obs["exterior_image_1_left"][end], (2, 0, 1)) / 255.0,
    np.concatenate((obs["cartesian_position"][end], obs["gripper_position"][end])),
)

# Plan
planner = from_config("inference/config/actor/planning/cem.yaml")
optimized_actions = planner.plan(context_obs, context_actions, goal, wm)

# Visualize
encoded_context = wm.encode_history(context_obs, context_actions)
latent_rollouts = wm.rollout(encoded_context, optimized_actions)
decoded = wm.decode_latents(latent_rollouts)
decoded.show()

In [ ]:
decoded[-1].show(), goal.show()